In [56]:
import os

# Set project root
os.chdir('/Users/fazedreaper/Documents/Software-Projects/Python/HeartGuard - Intro to DS Project')

print(os.getcwd())
print(os.listdir('data'))

/Users/fazedreaper/Documents/Software-Projects/Python/HeartGuard - Intro to DS Project
['heart_disease_raw.csv']


In [57]:
# importing necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

In [50]:
#load the data, get its shape, and list the first 5 rows
heart_df = pd.read_csv('data/heart_disease_raw.csv')

# Binarize target: 0 = no disease, 1-4 = disease → 1
heart_df['target'] = (heart_df['num'] > 0).astype(int)

# Drop original multi-class target
heart_df = heart_df.drop(columns=['num'])

# Drop duplicates and reset index
heart_df = heart_df.drop_duplicates().reset_index(drop=True)

print('Shape:', heart_df.shape)
print(heart_df.head())

Shape: (244, 14)
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   1       145   233    1        2      150      0      2.3      3   
1   67    1   4       160   286    0        2      108      1      1.5      2   
2   67    1   4       120   229    0        2      129      1      2.6      2   
3   37    1   3       130   250    0        0      187      0      3.5      3   
4   41    0   2       130   204    0        2      172      0      1.4      1   

   ca  thal  target  
0   0     6       0  
1   3     3       1  
2   2     7       1  
3   0     3       0  
4   0     3       0  


In [51]:
# binarize target: original 'num' has values 0-4
# 0 = no disease, 1-4 = disease → collapse to binary 0 or 1
print("Target distribution:")
print(heart_df['target'].value_counts())
print(f"\nDisease prevalence: {heart_df['target'].mean()*100:.1f}%")

Target distribution:
target
0    146
1     98
Name: count, dtype: int64

Disease prevalence: 40.2%


In [52]:
# One-hot encoding the categorical features
# No true order in the integer codes
# drop_first=True drops one dummy per feature to avoid multicollinearity
categorical_cols = ['cp', 'restecg', 'slope', 'ca', 'thal']
heart_df = pd.get_dummies(heart_df, columns=categorical_cols, drop_first=True)

print('Shape after encoding:', heart_df.shape)
print('Columns:', list(heart_df.columns))

Shape after encoding: (244, 23)
Columns: ['age', 'sex', 'trestbps', 'chol', 'fbs', 'thalach', 'exang', 'oldpeak', 'target', 'cp_2', 'cp_3', 'cp_4', 'restecg_2', 'slope_2', 'slope_3', 'ca_1', 'ca_2', 'ca_3', 'ca_4', 'ca_5', 'ca_6', 'thal_6', 'thal_7']


In [53]:
# separate features (X) and target (y)
X = heart_df.drop(columns=['target'])
y = heart_df['target']

print('Features shape:', X.shape)
print('Target shape:', y.shape)
print('Target distribution:', y.value_counts().to_dict())

Features shape: (244, 22)
Target shape: (244,)
Target distribution: {0: 146, 1: 98}


In [54]:
# split data 80%:20% / training:test
# stratify=y to ensure disease ratio is preserved in both splits
# random_state=42 to ensure results are reproducible
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('X train shape:', X_train.shape)
print('X test shape:', X_test.shape)
print(f'Train disease rate: {y_train.mean():.3f}')
print(f'Test disease rate: {y_test.mean():.3f}')

X train shape: (195, 22)
X test shape: (49, 22)
Train disease rate: 0.400
Test disease rate: 0.408


In [60]:
# scaling continuous features only
# StandardScaler transforms each to mean=0, std=1
continuous_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

scaler = StandardScaler()

X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.fit_transform(X_test[continuous_cols])

print('Scaling complete!')
print(X_train[continuous_cols].describe().round(2))

Scaling complete!
          age  trestbps    chol  thalach  oldpeak
count  195.00    195.00  195.00   195.00   195.00
mean     0.00      0.00    0.00     0.00    -0.00
std      1.00      1.00    1.00     1.00     1.00
min     -2.32     -2.11   -2.06    -3.32    -1.04
25%     -0.55     -0.67   -0.72    -0.71    -0.94
50%      0.15     -0.11   -0.16     0.17    -0.17
75%      0.62      0.63    0.59     0.80     0.51
max      2.16      3.32    5.29     1.95     4.37


In [61]:
# save processed splits for use in modeling phase
X_train.to_csv('data/X_train.csv', index=False)
X_test.to_csv('data/X_test.csv', index=False)
y_train.to_csv('data/y_train.csv', index=False)
y_test.to_csv('data/y_test.csv', index=False)

# Save scaler for later use
joblib.dump(scaler, 'data/scaler.pkl')

print("All files saved to data/!")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

All files saved to data/!
X_train: (195, 22)
X_test: (49, 22)
